# 🎬 Flock Clip Engine — runs in your browser

Turns one long video into captioned vertical clips. **Nothing gets installed on your computer** — this all runs on Google's machines.

**How to use this page (once, top to bottom):**
1. *(Optional but faster)* Menu bar → **Runtime → Change runtime type → T4 GPU → Save**
2. Click the **▶ play button** on the left edge of each gray box below, **in order**. Wait for each to finish (the spinner stops) before the next.
3. Box 3 is where you paste your video link and API key.

At the end, a zip of your clips downloads to your computer.

### 1️⃣ Install the tools (2–3 min). Click ▶ and wait.

In [ ]:
!pip -q install anthropic yt-dlp pyyaml faster-whisper
!pip -q install mediapipe opencv-python-headless || echo "(no face tracking — will center-crop)"
print("\n✅ Install done. Go to step 2.")

### 2️⃣ Load the engine. Click ▶ (instant).

In [ ]:
import base64, io, tarfile
BLOB_PARTS = [
    "H4sIAHo7S2oC/+1963bbRrZmfvMp6tDrHJMxCZHU1Uyzz5FtOfHEFy3JaXcvhsNAYFFEBAJsABTF6KjX+TUPMGueYH7Mg/WTzP72",
    "rsKFpHzpSTzdbaI7FlGo2nXfte/l7Dg7/3Hq3nyn3ZGOv/pNnpY89/1ttXb38t9Ib7c67c5X6uarz/DMk9SNqfqvvsync6SmqT/V",
    "vfbh0W7n8eO9x/tOq90+bB+0K19tn3/6xwv82VCHl36od4ZDP/TT4dCZLX/1/X+wt3fv/t9v7X3V3u90OvuHBwd7h7T/9zqd1leq",
    "9Tn3fxxF6fvyfej7P+hTrVaf0hJQJ7wE1F//63+pRAfj5iRKUj1Sb2bzxJnFUTNJl4FW1zpOfc8Nmlg2KtYhnRk6diqV89S91ElX",
    "+SH9SVXz9yqN3TDxYv9C4+068twL/Bj74Wg4jaY6TBO8e3POHetx7E45q+fOUj8KE0lGDU7lxPUmahqN5tQGP6FaRnqGT2EaLFU8",
    "D0P3gr644UgtYj/VyJHqeKpHvptSOjV67HpUYRqpRRRf7fzu5+ji9zsqiajMspKg8VRvqKixfpjMtIe+RzHVj9VBL06FBqpSGQ5p",
    "BBJq3XCoeqractpOq/oPjSedv4vzf3f9/G9vz//Pcv4fFs7/3VbrqLPnHO23Hh8dbo//L+38n6d+8Guf/R9x/u8RASDn/0Grvbvb",
    "pv2/u4f9vz3/P8v5fz5xYzruJjqY0eHWVV4Ujv1LFUTuiI7zhhqPpzN9uTMeEx1AB+QidmfI2FBJML9MHByM4ziaquFwPE/nsaaz",
    "0Z/OojilwzWMUpdP80rFpP2cRKH9HWv7K5lg8WVv8wuqy9NJIpBnbjoJ/AsL9pReM3hLdxpUKk/fvH7+4tvh6fHb7+hgRoYaNccP",
    "qDF1h07xKLjWtbozo56GqfmjdlRV+lrFzwsiWEYOwNFBXxnpMY/AUHLU0IQuA1b/qZI0ploKddZBrIx8L+1WFD0Ln7JFRKFwsbpy",
    "EzWWL3hiTaMUcsOdxB3rIeqpjeumVoyqP17WUn2TdlFVQ03dm0CHoK5SqvegxdXRF4GZUFqsHRq0Wlzt//cfFz8mzUG1oar0H4A4",
    "QbTQca1ed6iIP6vVN5T6MRk2B49QqEn/JJLFNDTpd6UBAwOA8tRBHVWBPOxgERVW86ajrgr8JO1TxkFDff311ULamk2o8zSazgJN",
    "JNWpJEgfaBGdzUPlFnI2VOz6Ca1AGU6QhXOs0yQlQiVWUajGrh9QEq9AAEFB6lWhMtOohi09jObpbJ723sZzLYNjfnJLGYg/ZjiO",
    "dN6LRlr9S0+1CtNHrdKKWotj8ySOo7iWfcMzpkU1nYIWRfuowbXbFYB39a66fageOj9HPrevfvdjKJmkc/1mh/Bid3BXzSCXZgRZ",
    "zbDznhyOqCkh6NKkdu2PdCQrlYc+ndNw92nlNLB8BvlwC6jawh+lkwbtfv9yktK0jlU60USmx0TGMywsNO1OichHyVOi1GnzK4MN",
    "vqFOBkGiLlzvCtQ17Syesp8Eaaim/5OdsMVEh7YYEfHhwxS0dkrF9YhB1xJiCwy2URdzPxglhBeIz8D808QpP607tvl2rgRxOIuJ",
    "701qVQO+Ws+nyywKrIR+aZ6yzFj01/hXYzL5lVgg4gGG0vMESdfdVrVRLt9MJtGCzk7aE5rzSPaejKgMKEOLxvgDxIe9lcYyR/Uc",
    "3KCe/RQY1GBkd4AZkppdGTQG9X7VNmrQbw1WcQrNcE2+96vcjOqg3iilmmZRjVz2gXpOE4DZ69qRn8U+eDPTED8cR5hYM4lY1vrG",
    "p+9hFDZ/0XGkamGkZFspIFwzRZv3Yr8qlfCw+KXRGHxok8oYTQ3a0m5MEx5X/8DL3fn632s/jm47jf27+k32i+AXNlW2velMUtP3",
    "7mds4Xkw4pwjDU4SvHG+ybBLbrnZd9XSxsRIT53LOJrPau26GXqb0KlbFM9M6hAzXDhXGiq6+Jm37Oso1NI+fHUkN8ahxotiNJ/O",
    "khplbjAjHKa9TkNRyzB0buL5fo/mNNFZbTSPo9XKzAYxzS6sNT46+dRyuBxXS6C2lPI/57Pl/7f8f1H+v986cg4fPz5sb/n/L47/",
    "F+Htry8B+AD/3z44PMjl/wf74P8P6c+W//9M/D+Lv9ss+38hS0B9q4l7JwbYcwOVRPPY0850tqeYG3fVn6L52zlR8T+cvQQfaDOC",
    "/PtUcYBh/O9n8/mLg0w2nQhJQ9nIgq1JAw2vDPl+gf3Bj1WeB/WApi13ULgdRz2LFiHTQqAVf5JvP0Hr4KK/GbOJepzp1ciPayJO",
    "SAzBSuRxkg6jqwLRCualxyUgaMjHs1rJ2BhJY21DAn6nVqtO0nTW3dkBrYyfCX7XC5zNA/UEupZXp3tqPkN/2q2j1oynkVi+MF2q",
    "MU2OVdkoL45mxFRfaT1LeKqIxndD7eSE8DqLtEybo2C2xvgwP3NBlfOQ9YWn+F0P9Q/6oNepb4NHyODOR34kSXvuYAdJWQZ+W4Pd",
    "nOr4UjeFB2hSD6Yuc1EYrtW8keEgwBiVv8l4rnFYmujifACT2LPCIslfd/TNjFiceQJ5SZbP8AyU3eHJTWqFWchZiOc0pq+j9Hk0",
    "D0fCR1CJEhRAyIRRECtQw/OEMkzD2HrRbAk4DeQt8Rr0/isc0c7W/mNr/5HrfzrtvQOntXf0eK9zsCUAvzD6L1fZ/7o04Pvpv/bu",
    "3sGh1f/stVod2H8c0J8t/fc56b8OEw5vc6sNlrq/e3P2rPny5A8nLxWwBI3VdJY4lcq7KB4VUhRRQCI3xuHHJJ6VJC8mUaCVrLAu",
    "EUeul7LRxyyCmJHoETd2oyudmX00KpAxJjPtXum46S4AWaxDIFd2g0CJ7QcUAFdhtGAFAUTLGjYiC7TLB2Xjj6iZT1zvivKykqFJ",
    "NERwU5XT9a//43+qVy//qN5NfKqKyMZvT39oup6nAx27bPwRquMZkVHq3A98j95qr3TqBnVHiK4IuopXrucI4IWAIegAXDCgaahQ",
    "ayIln/7w7JjJMcqmR9+A1koA5OnpD0yUUA+onzU/9AIHcOsG8IiotUvquwBGI8ex1kRFztxlk2ps0qjMU62OT1+oWgKBbaDHKde0",
    "jOYgConSo1ZXKm9ERJt4Ez11VQ0DlTgQ/Bm647bKSdWu6stP+lUdu7QIxvMg1ImRcBN9Sh/aHWd3DxLzcCRvj4/wVSYNBc9PT46/",
    "PzkbtlrVu4ZyHGdwV/lEziBKPokryGWqhjnIsVlRJ1JkEBqstqA+MOug/pMFr0QR4k+ZnITgORrpoGtUf9XABZF63aFBGflu7P9C",
    "a/siigL6xmR/mfV4oJ7psTsP0i4vunvWFqa33uDlYNflH4U6N80k4PYXzW+UOJc61eF1rfr05YvT4cnrb1+8Phken59hptx5GlUz",
    "kXcGoGe+5LRmDlu2B+Ue+snQRQOHiTSQiFVuVb7OKxv4mnxJVTfVC+h5tZyZCg8pWeanIUPc438tuV6A4dM40YKkXcU6Gvk1NC0q",
    "anwy0OUsH1cLWpr1cgNM+20DtGwp9Mzfe8Bne3oDePvNKEU2MC3Ca/zBDeaZsmIeAhGGRZO7bGHfml9WUVFQPdDsNQrbnv/erbEY",
    "spk2jqXdRfnW4HXPCmAow3Nlo5RumtJ80gTgGQk9EhtOP0MYBO7IuMnxQVg+aQg+L5wzUAtCb2i0kd9G0Yhy8t5ipPc0CtyLnXd+",
    "OIoWyc5LQo43mfrQVYTJgmxrWe2jQ9swvKTdaNBXWcXIiKbceYtxDKBX6L00Z6SvfQ8YpOrNR67spYmbDPGW7SFvNhfQXjSdAXdT",
    "9nEQuWn7gEtYIBkUKUan5ZGUS2hpUaHbHAl1SwjJ/t4tpO8yC61H/nyKVPPrDhikZhavTTSaQSjaij2sodqGaV1P/jRsH4bpcqZ7",
    "5sXYF+hLNvJsKJhKTp0COi6oQHmCh/kEGxnKtTuC9QYNupGiVLJd0i2sLwLcFx0oH636EkjCVpxvGXxcMP6gb47sNUrqD1ZYbkFg",
    "sG4JR7Xb1QNA2VNxwSCsMUVjPZ89JWMQQzWe29pCZDvU491NReQgLRegtPuybz5pyxnLW5n7ZjdzhnM/fgcXiCXevh+kl659IpBw",
    "rJmt+jpSxe1dS0qbTrlJMqcFWP+GZ2tK+9kHfUN4iijFkUcbMFFCJIGuAVElq8wUp11P26Sh5ok2VKeg6QzxMjX73fPh2zffn7zm",
    "ra/d0Yo1gWxrGhy7062O/JU7I0TgE5xgKWOlQpeWLOgrDAwMPuahny5ptGdR8o2yuw6kKJrz5zkhOfpsUJVjZmYWYR9ns1ba0NSM",
    "ZgZ3xzSoaXM0cZo21krufrDk7mrJAlLYWE6+F0qtII0PV2Z2b6wTYOlecYCLWKFoApGhBxB/wygeTsZDDFcP/9yDNCq5mczHIQpp",
    "EPemapEGockSWshQBzAH55TjsrGOPQh1Lwww3v91TH6uyi8+tD9SOpr034R9+vJr8KkIqG+SB5+Ag/qcOPiN0NAGMpOQD8jobnFD",
    "zgI3hRy4ZAFl0pxkSTt5SiVxXj5zY+IHq2ylkuWYEl9IvKfJ4sbTgz1rs7ZCza1jwxXqfiNytPyHxTiELySJsGLg/uITykgixgIx",
    "c47CFgtDLHygJVDSKCYWNjeK+kw0BRd0E/A1thNsDDLkcbA7/cMnPsv614BwauHIr5eRAdVbRAKcG3xZ6k2GoDh67YPcVInGhwY1",
    "oxfFQ4QYJ6yqZqCvCS/nWEHahLxD04UpHUfr7ctz1AI3vJy7l3oIK72etLFftanVwQr5s9KVDC6DrNniGWqh4mvNaSjTYzO8a3us",
    "pGbgtT/0Jm48zAYhMdY+leJ5CF7i9s5yYWYN87YocI32NCzyTpS12JNn+Yl96s+Ik6F9REcsTSqh5TSik7W3EeDGgTLwMxKghzeZ",
    "8XrBkG1tNJMEg8Y433avVoDTMGU+iUhcn5y/FecbrMoaIiqz6GKlzkOeH1qsPEiJqsVurOviXnXlz76AE8GehSahUToi6h99RpQZ",
    "4oJidwON+jpqEllKm4z4hpDWxrV2oJQLeF5A/wHxPjMA1fmz72G7eO02OxmaX2M464Z6fQmpmpvQFlL6ZkanlZ8qFrcZ1O5NIiDY",
    "KBc7sTQzIz9DQk5EpdJYsG8aF7LYIzsTZmaL8TJBgWU6gXiTeNnQmGNGIRzeNLTTubilTMaKgOB1lL6AoTVWtx6tmClXs0HI2ker",
    "d+HDvHqpU0f9kGQihN5DuxMfNsA1+RaqAjVOJ38O1c6Vg7nRoTOLKYdHzdSG/k6mtNyGotN9ZIb80Tyl6XJDTyeMoKZEa9tJoM4n",
    "TtUQdmZBFM6+ErGQxst8b5ozmQ/UVRtZTnQAwgEBcu36AbwHjb5X33h6lqoT/kMzuGa2z9j2n0c1ttX/bvW/Jfu/3SOntXd42D46",
    "2up/vzD9Lztp/wYOgB+w/9s93M/9/w870P/u7u8fbvW/n1X/u8tUx3Mo/HZmcTSae0zA8JqYB268JBo1puMcqVAAG1ng1IUYLzHy",
    "MPbAj2L22oHbBBG+1ePEnU1A/z0PIu9KvXWDK3qDTz5RPlys7qh3cDm6ARUEl5MKi/BABBiy58nJ8zdnJ3logAtxtuAmI68RJl74",
    "lxx+wIrbFpQlAuGFMAYNxU5CigiIShgpIiddSnFTAEi0UcEQle6oZzGRjkToEUkea6K3LpbGJ3In9xHMghoMqd1z6MQ/Ub0Z608z",
    "erROHI11TSeEKMsht0b0uvzRSjOk6V12T1zRS45cZohzB5G8tFBD0jfKIkCEpC92m2by9o6dAXN2E7SkfF0jnnL4lqc/R+dogGeT",
    "2KVZCAgVEaE40okH0lrkwLwSCKLGchrzKkqxiur5BJsFIDSx6CZjxKaAXjNhWUzNzBON3TQhHkld6WUvcKcXI1ddXXdVk6quXV33",
    "W8RGEYEPp5eCMBHMKg1W37CCwk/CN2ZoGDxiMjM+LOcgOfegknGfiziCR20M3Re+m0bm4wSuGA3hjFlDSpaIkOK0y2xoub51eSNk",
    "RiFR3bWsiSwMM2moqt7dKHnIu0S5iZxPdHythx7NVA7KdCdvZFlfaSTo2UR2VRLQ3lMuJo84G8YZseZFLXt55oZipgreB9LkEjtQ",
    "5r+pXa0yzzzxESNE/c6MIwaEVr0K1zto6u+pqqqKK6YMCO0SDChPUGGK+37XV49UWHDWK4p+DbCPGdVYzwLXEyaqJ6Nnp3pjfrTm",
    "Z4g5ZmiSDudTVrrUCnD63XBwT23ZGu6j+T8PClNKADYWoYw9Fa59Wp/YUoF26dOnbY6CWjrHEQ3ec+uygSLuk9FOct1V5hZtSpQ8",
    "o61bNB05xinagikv7ogmxQ9dKwQuDPSGimj2bX7iKOdwji8a82btyGA4Js9q6X63Pfg4CJzT5KNJLX5pdweVzYUqW/5vy/+93/63",
    "vb/b3nU67f2D1pb/++L4v2Jsrl+TDfwA/3d4uLdv7H+J7ztA/Jf99v42/tvn5f/2hP/zYXwL+1VoJFwfKgyO9WZDtrEc92ngzkea",
    "uMC3E0RjExuH6rUfC9uVEKeoqyLBhkUlpNhXnC/RHp1LKnHnHvFZL9KHLFF/+fIV8yCg9yAXz0XohjjESYaPDHjUWG2gVeJwDy6g",
    "SJ0jGRYowrf0H0LLpENPPxxUJAYcWNTmzE1SifIWhZBAW7sQCOQvdahj31O2V5Z9pF6wETPxhWKPQpXQGoqmFaPSvdAuNYUD51Uq",
    "4nGW8BiawTv54/HTt4o1LzuaR5saOUI4nRCqHqiMcq2mumCDakSYAD/kzRF5AQNb4VlxVbIMWHz9yfxnMQ7PJ5ra3sOLvnrz7OQl",
    "0Xr3GKbyZ9BfHq+eJpUJddrcr1oarIh+NvCxRbvdIk/L4XGGPMBZhJwVPlfcFsN0EkczmlLTjWObUCnxeRu54TLjZ5jcFW5rQyCJ",
    "ahgZuHA3JBJv+Ytod0b+aEVTFM/Df7emMoUNQPxEpo3So2H+xbA2ovUHEvfGlxmj3ufQPLa5YiNBHwsaoD9FczbapxU0ocFgRzvZ",
    "6UTXEnNx6cMNFLnGG1CCxQZ+yE6UtDULaiDR+chmdjFKCKXBRsIuworQlnYDoIXcMxFtlcoM1Xr+9uwFbZL/dv7mNau8MiUQ/oV3",
    "HnVmTAv+2Gzrrrpd3+l3lQqjM3GOvM1WyZ3ZjWDvCG8VBvtCE1fgKI42yWPx6ofzt91KE3EhL3S60Dqkisxg9x8SQzo0u5dqY1xV",
    "/Er1Fb6anzxaBBHRocSLYBJFVzJSgYsMQBJ+qR7kGE4JyQyRYygZNsAmsGaK1JvXJ2ypgUhLRtQF3OMKZiG0g+mpYRFT9/0Ab9F4",
    "XCcIPEGKZ6iL1YHhuaQB5/Ic8AV1EHOn5ojNydlpLFMabEKlOhs6mUnsjkz5bIdAdMm5sV62wMvRh7DAWVFs9NCrxfNix8/fnpxx",
    "ycAtFUz9NNBVoIUEXPZOKgFcGrx9U0YGtd8dtBRMKhIpgsGmEmyjw84qmCmMDzSznGMxWVIGrOAEylWcGzSRlEqjYM4ShCDlAXv7",
    "3YtzZZckl5bDsavazXZL1XjyIeRksdejbM7gcEGvtmR+REkjr3wekapouavszkEnrBcFI4nMRWzfPJ5FCb5xxCbNClaWr0yiRd2e",
    "TLzFaPfcGoTRVX14a6jB3Z2Rm0E4x21moZwDO85ZTIBx+p8dvz5/evbi9K2q8cQH7OPT5/lu0qQNWApAnOxtvsnuOJCqwVoiAckQ",
    "cS2zqJkBlfF3hyY5QXBZh0gHSD0qJY+MHh8rcgaItKG3RzRlwyC9nvzJzQ0stF7/thpHvDiqQCh8MGFhh1ipSLkbFE0KHyi2gk6o",
    "oqXSUz8VqgYL4wISyYQQxDgyXlC8PTiZkT1kS6K55w+mmswINFhylB0Ioowcio5PN03jGoOghuEzSy5EMiWQxYpl5hhw+UnAjgQI",
    "xGUEKnkV9fK5YDP3hyA9aPjYyCapeWIGkUhtHmrirCulvJXPOBa9wWYB1AM1ohkmlNFkKgcUSw7MwUqrFeSxXld5xnyEd0tDteqQ",
    "wiD4ri741dum8F+OUjc0Dd3gniJV5e4p654QdgvIKXGfJ8T7juOi3VFDcdgnS5O0O2WpEfYKEX0+myc1jBQzk14WhJc5geFN5uGV",
    "9GhkxZFcx6BgO0tbD9GgIPjm/P3WILPzaZikZntg7HgqRXFdURS6QV7HZXMJJXfAWieNq/1brqTrdMZ3zVv4nuDXQN0CsvU+yWSF",
    "3OziAFd/DE3NDDeTy62sTJoZS/WtDXgWhNF4cmV2QecEg+r581xzTOuc9BY7HCKoXTbDIWw/X6e7s4gPJQsTBgJjZt/MU0mhsKCj",
    "8yIpWFipphIDKvQgt7uq17Pfufic5+5DcHn2VqCKZRZg5nNr7Flq3+slk6SNguvQuogRw2YpXLSjafr5O7V7f95Slxqq0BZQ02w8",
    "ZpblLnYxJ9BnvBrUSsgOx0em9cOcigwe/KVQpVGm/XNmSzWKNEcQjImP4FBwhZYYSp2lzivSZg7CwD1qqrZu7kJNUJik33GnH/Gn",
    "khQVgDNJcY5ZsxCdK/E/zWbi4JvFmJtUu0nLwnxUf/rpp6IBZqko6wQkR789cGI9ja71jM5z/6Ym8QTLMT3Xw7kx1q9s7X+28t+P",
    "kv+2WkdHj51O5+Dx0f7+Vv77hcl/vXn6W4T//mD87/2D3Sz+957E/+7sbeN/f1757z6zKk9pCajzALbMbNaMoxdktGHMTVgl4olx",
    "ErsBGyPToQQ2iK/coDTiE92Rcn2+FoSthZW5NEOYZhNmlo4tmDMbbpFJ1oFTAZcwtJDgAhromE5YHIRZgx4m9jCHITUCKhDtAEea",
    "hjAZl+5MxCwaBIQbVn66BA05IRpvQgzyTw2W1BDb5Inxj5r5GsyxGBOJQFtEu1N/JDedVOfTaVLdmbnEGCaGTqS+X7uhn0w+XvT6",
    "t0U5o22ZhTgz4s8yGXyPWLTU665QirCXcA5XxKMrfBItAY5Nts5P3EvoZXHMVsSPwsRtmlRiZXkWKF/Rx6JACWVSfeJmZZKHmORa",
    "ichb6eXGKF85nH4t7089bzR4LZurvmJeQ4sK3lT2cynm8qaQ0ll046UEkoaF1Lh6m3Sd3fEdJ6WRJOk8yUZCNlHIVsOceV0OT02L",
    "5qZzsMclvK7LUR5cj1/ZeoFFBNC5wF19Y2S0QX2ViGe+tmiiYzralf3JUh3qZwP7JDR7xlHfcwy5450/sJfMMvRgpiPbyDHQznRT",
    "S/T0yJYTeWPTU4hpVgdFn00Ny8oKt/qob9+cJiI38QLtQgJs4iHHMpG5Q4/fUDVMUr1sG5PNZyESeL7Ix1VAGt763VZndJcv999y",
    "WvNAb7/mBM82gi1MNA+ZZdNnRm7C88H3IxQ3vsxuepNWzSQeXxBouBMCZyUsRR3JwcAcmvV9kdkd6en8Ros2jdqT0I9LNx4FkGTS",
    "8eEtjLt1ofJiaGsr+xpXuVkPbws9u3v4Y1jl2WYLKO6SdSQszdb6TI2NYM+TKIJNXLiAH63SDBXbtGk8/1/34BrQwX3SpSKm2yRS",
    "cmcGk+dOUBLh3wjF+c8g94l6GgUBnYe6eGQaZ7NonpRPz4tYuyzRpC2b0FCwyPj3qNPK3jNZCAqYlvE/ptry1txkkuiPxeeVLYWL",
    "4hGksmCqPQBXjo6WYywWvjPTXpZfrdumoYBd9/1F4dyyZQdW/JBMEVZq5o5YK0KreqHVKIJ4gQ9/DAuWv8uHstKjS50UZ69fm7o3",
    "NYicqRstp3WEID7qkfwUp0Mk+KzjSwZ/X2zVNv73Nv53kf9/vL/vtDqdx0d7W/b/S+P/zSWMv7oM4EP2X/ut/P6vfeCC9h7l3/L/",
    "n5X/P2D+98zcw9k+6D4GgfG42z4wvrsm0k4aux4TCTXQfhOi8JgeY4MnmIkE7DN9PJvFETEPXb4/xRM+e+x6WlgKqaT2Chd0IrAA",
    "Ir34cOuY4OSd+qFLPACyZ/EmlKtSBJVJK4hj3bxRHnTvsUQe8onwSaY0QaJDQA4J8s2+IUBvqqZvZlEIlx4E/Oa81AnmQvxUXcIl",
    "IakUzL9+9uGWzPefMesT6NRKLtjdRgQHOm4ydNRIvbZ36UCvZ5w7QJO0WWvOt5HCxTuP8KHenCEAGjqaSOhMIV1YT+imvmd6KT2q",
    "ucHCXbLw44oNfzqOer5a9P0PAHPf9YhrbfJswsaPKqhUTnO/rzk8uUdC+XdVsiAakGUj5hJWUDXfRhF8brJJVF9/fTxPo+eEUb7+",
    "2prLEVGVVBIXqnxvSUQRselp0yjXlorDe8LWwVFsSsikv7EnhHv9NzDUWPWB9yawSEl+G7HL6n1aDRHEvPnh7fBdQ+EPLphDpPOG",
    "aj/utO4f6we4UTYK2cIpN3Vyw2s3UUQxpt6EZs7qtPK7enhrsJBjeG/Yyve4dJVlORajZwwu9QHtX7s1LK9PiGLZa0N4x+AKI1Ha",
    "fae+Vo8JbPugXlf/pv7SLvYVK5SvnFK1MeK0wPwz5HVieKZ6Jm/JgP++p951379iHyg3gEXeMr972E9oRXJ9vDPhAELr6xdd5uDX",
    "WcGM2cv7uhLBotq8HjMXD+i9W+hgacBsc+t33dvv7hoJtUH3bnlFUAovibvqGiTDHULWUZDFvEcCI1udxV1mVw5NUqHFDZ5BuwJK",
    "V1qZzMX7Ac7XkcjYoKgG7XDuSaYFz1erLNIMzg0sB98Rb5MNhNrZUZ1ffbgteB7m7u3NXbf1mw32A/UEF8zhVKFN0LwmZAQsw0N0",
    "06Szghh27AwxHc1wpp0jmBab0yAhXtKbjqzcy7yasL+wtwvYRotRW8inHm5P49OJNckxYodcaY4G/LD7UOnUo9NQmFCwmzh8DWxv",
    "MeqJHSVbRRurLp68Czdh6A1rQhLSobljrmWjs2QJuwGW5RghzJRtXTPRD84vShMkMXGDMXTQdpvayWbLjDKLTwevd8PGIaurD6vG",
    "8MW8jwrrh8s0uZr6vfYkt2JKYqZE3d50nfb47htjSIL2l6RHK+YjBeHQx6zMzbK5bH2aSe2Ne7dcMUb6rrFp1bY+Zs3es14LjYB0",
    "nSYbHzBF9XttkVYQRenEeMemR4QuzN+ydmA8S4aJC/VMrhrYc1r13GqGPwqpRscgmwGXaDMz50SLQforsHD9SUZnIfJZTpXxXXYb",
    "bWnM0etdd1aTcpqJDs3p7OMitPSNZJ/OVigtrjsOXxL4VG4XXAtFlsTecCxmZK7oIlDk6fHp8PTszenw+ek5m1TutoxCRM/Mym6b",
    "mxVN+Z3CkBrQQvtGsFaezhzef2zAgtEbykd404OIe2bfJN7aUG6gpPde+/1Bwd77wEI5q0eush0BZ/Razr7pvSWFYVEttkRGhWNo",
    "B6vEyTLCGs852q/bsI7uIkcJ/ugm88i1SHPoIe1dhkUEQcGIL5+5iM4j4QlkFnDir1/AEl2VJXwsryxmQvX/aqaodGcrLznQMpRh",
    "x854WVrImM1OmGMureSl4F2nT6MAkZZjxrC8PN68fHM2fPLtWefs2yf1+mpQMILmZOO+wSH7Qb6TehIsNEnVRXSzlhFX5JjlVob5",
    "3kVRsBgb0X53cM0RrwD40zr2SBoyGU4zSj9uHKbd1v2P6RPCjuNSqI+Ash7XjIkHwL+h/hL9n1WF9VAngvLdWpnywskX6NelL49U",
    "rQ37tJyV+5pqW7mWZ2HPk1rGIXLpoks9LYnMeVnWXqDhA1zewo4XRFmi9a11F//wQrKt/Hcr/y3Gfzpot5zd3fZhZ3cr//3i7L9y",
    "A9vPKf/dP9xv2fhPh61dvv+nc3iwlf9+VvnvIct/ca9P82IpIVdWr+YxATCPz89xrzf7Vsmdj5XKO53JaSbutS6FbTGhNC1Ty84z",
    "gMFi5R9/vMrqkRAvEr3ZlK94oLzUxL+c8IURxDnPWXLKfLmJSgQJk5skqpaz5XUOEQW/NDCnozkkuXRsw2MNjksSS0bBpw6+fWcn",
    "zSfH5yfPMlEI21fl7RdPSWsMYfwGrS0cOxyj0Fz0xRWx7oa6uMWBrTBWsevlBmzWqgodBdVqDefEMkotaJhhMsW+wJNs+OF5VBnF",
    "/jhFBB0aC2KvKJW9L8URBhOImPyzFP41cvtj+xuGcaan0ZpcV2z3NI+SDXb1a0hVmTfFTR2joZusOW18jBTTsG7WmMysP6PuXzFX",
    "s0iLqqqWHHlXbcs4fm/uL2s9OEpWbcLTTJgYgiSOgA7lrUYVmdJz5hWG7MMyxJv11KAsxjJtSAsYn6qmxg2iE/rKghMqv8GtBJIT",
    "p9XA56IHD1G8eePrqz4jVEgAwPigI4VzX597yhqPg6HsDjS65uEODupMY2OBkqCmHIeq+ozY/uiSWDvVatwO00TaU7+TF5jd3TVe",
    "uX7YaLT4fw3jHpSByS5MLUp3zHw8WvMSMkn3iUdWZ6hsS2OmiKUjuTVNnmNQiiPUL3pdmaLiHOWzaQd0EhB25Y5bWQ0wNZQGlce4",
    "3BzPzXZJPupFY5/MdYx26JtQ8wKCZeK1jo3XUvJN4RY0WvkJo1fCfhJwmx17c1yqBLcSBnpIDCPKPMzkMwWEm8tFqlnqkIsWt9Rm",
    "y7xF2SgPY1GINT6nkUjKwhRRNBQdnApRqonDa7da9XpdpKFAWdYnGPVRX0sXKomZkAmNbYMlwUbVdofTEO+pauLoZ9kLMvTvNB0e",
    "CcToZpDyselaK2PjgmkwD8FmnAtUHJpRdopiCl4ieSS23/fUAc9NPuhliYHMU6/wve+rf2UweVJ9UCpTMvsbV29vaXRuZcTvfvzR",
    "u2WYd3d3t2jFHT5TGqGvh7PYn7rxUmb44YCyqOr7Yru9ryILXpV3qDX1Ezu+zIPJ7JEyyi0cD9n6v4gCTG6zXZrNBd9CLG6yNrqX",
    "XBp2EbjeFV9ZTAXx9zFCkMuUt/JrAvhOJZFpVrtqn/OnaYS7UTpwwo1m9OtIrj/Jap1FiY/NxeJcKUsbf7/UX0Qq6J9LiIEX4Tga",
    "VOTl7XJGOOB6z2m1HlVOA3d5ppM/dlm5aF//1GUtY+Vd7M7O0yWEtZ1Kpf+HvUeKX5MB3O6nLq3F1yycek5LMcx+yZ1NpzKlEGLN",
    "44Z6M0+BPe0r7kjMftMAZRka6nzijqJFQx3bywYa6pUbE6/w0v44sz/+0FAnsPcluqpiGsp4XhbVmNpCa6lxiz3OKWjZQ97Se3eN",
    "zSuv8W/fWZEA/TyyP28xi7ZMJE1l2AKXm8zvPKl3jYMW/t/u0H80cifXsArOR+2lu8T1j+dCkZ3ALZGb3zDj+b7+jsca2PotHVFM",
    "NVmhPJ16gpY2Yu+J0ala1LWzo3YPCK1lt27hY/b1X+Ujch2YPAljffv5oFVeareTu+7tlO2au7dJt7UPLUq18vct/9n6//1/k/+s",
    "xP8+2N11Dju7h529rQHgl2f/x/zxr+4C+AH5z0Hm/wf5zwHH/z7c293Kfz6r/OeIxQdvYpglIQIU0azqCfu+24uZ1dS/yTz/IDsh",
    "mvgC4dhYojNGBFP16nSP2CPcbMRKokrlODBeIkQ+mSuasmtO6CSOl6r2k1G2/VSXMEjiHxAEYIYvtdD2HG2HBS6KdbeglyvMIcJo",
    "DF54JlAdR9p688PbHWNRxr58AhRxAzyC++mxuj8iPlqmxC7IyuxvXPi0mm3O18nQnw0fi4HQkMu8r2bzQ1Ybuon5tSGL0eEij/m5",
    "IVPh2ljKl79tyMqhv5FLYoDnGYomdHytlshuGuI8lgTzS3+83BC63OpZrX8jh9slom448uOi5dsQ288mfC1XtGWXMXMw8LJCdD0k",
    "XENWrMB4313P0vKhMXxcvRc6FxQAjhERSBGocvOu1wpw6rlrJL8L61HMnIk90HFnekX/gj3i+73kSlR9Q5UOo6tC+JufowsjCOPh",
    "gWZbBtp6nJkAEjvVOsRAHBiKylSz0h9TkXD3MYjSanvnyKy1v/7X/6nWv8mkVrHHdKtdhY78rFkRKVVVL8LpEJx8neWwrOSuvAZL",
    "V8fG3jq4XQInK1NivFuA2SJ11iLk5xZ01oQOK0ruqcYyqa6G9XvvfUpykSbK7djbDxicDwuw6w6tovkFo1fCk2IYhiCRTXhOAm3N",
    "01QcqDPr42pl9VYt7U2aMPKysfcYKVApuaup2eT6sBEYXYbRwinK1IqjtUejxcY8BsPY0eLtwnsTpiAG5TjrQRl5/O34FYIvrgSg",
    "YhOOHOZK6Clpy7gqsiJYL99CnCGATIA+Gw2RTxYru+NdtyZnYql6WdTEkAqxk2ihUzFsGbh/Mtlj3D+r3/BXsxk27LNig/dp9Kyk",
    "X936d6rGgfr6D1kd8nDQ7+61Bnd1O6hiHWPQvMOe3FjBKNLgWsumnMWaDqgmi79tbeVtVw4Sj4wjjl+ZIXrHmhNTMz5U3SE6Zg+v",
    "D9cHfUuvdMI5ZYn/h+o74u6B4CzXll/Dx9NuJUtD6HPEAocoDqrEDqLBmYUFiX1YtpgTUJY/Z0glSzkCt8m1fuUcKtnPFTbo+rmy",
    "os0AVgbPbpCzKBwkNOGgXlBpmPp4gd4id8Ez+XrMIS+t3i3p3VKjxRDRhtObYvbfY+64aumYxTKVDuRy2YuC7oWvZ6wO+oLehkTv",
    "DUcXBQEpan2EaptinT8MomjG1bZLlXPxe12gxSpSbvweSgzEm+p6lnG13+peD26vx3f968E3/XbXHVxHAe353u3o4m70pH9xSckt",
    "SsYvlwjWHlvCJr1O14rBexxdsu8OqpvaMHW59QSem29fKfdgQ2yBvPdsKXo9HlTKyRvchnM/YXP1MvsSx+yo3Fm9PXfNy/hCXtsH",
    "ravNlqOZ4Ss1IrcixWKRY97a7ZXD362uOw4u1SjeTW1DaZZWbyOLl5kr2zih3rBxMgtaOHov+lTbSJh5DklBWRPgMv/ECaXSTFsM",
    "bWzRldgU9rPOwBiNQgECVNcI5EnjxxLTyvpVlAVdEp9nM3MBZenmXyZnN4ffQubarM72jKLKqte3/r9b+d/fsfxvr+Uc7B0edvYe",
    "b+V/X8AjcpHfto4P3f/X2i/I/w4R/6tNqVv53+d4HvzLzjyJdy78cEeH12q2TCdRuAsB2VNQ5Sdyp8HTly+Yf4UchA2ORAjnhwga",
    "4oYjIv6WsMoqh3Tno9XwngzXSuGaTabKiHhI01nS3dlZUs65c6F3Xjyrgpl1b5oCYf89pf+yM6Y5cS/1jp7tg1JGSZZK/kUu6tsh",
    "apXSdxHs+s9zn4ikrvXaIminx2+/a6jj12+/O3tz+uLp8Pj0xfD7kz8pIs5wfTlVzqKtWIrKpSjpTfrJwkM3vmRHrw+JEAuSeMcw",
    "RrlPKoRkhhKZun4olx/nQW7ZXMpW5BzHl3O095T9y2qI221uGehVzxHIfxJxIFy4bMvMCQsmlRsWzJ057ojYOQOrVjXDXm3YIRkZ",
    "gdFEB7MeLhR4O7/Q6oezlxA0wVkgMMsEfb4fasQwqWfuPEh71Tc/vK1amMZ/DbQpGP2yoPd+iJCIFUGad4FJYwHf2wwoW9+xn1Gq",
    "3wMT8pUiTJE5mnu/k17fJhjx0cBWlwm7GcD94O2KJxDpcqZ77LFlazu4vxzXludkcaYdvTVRvaoRR0R/3FS2rWp2WqMn9fubJVzg",
    "PfBZkplGqnApZs3k665fmJnVEl9CckCVSSBdvNfqlZLwyEqFc7M4puZ7yOvwKszJeMO89JjU5gzlmGeZeLSQg33qGqVA8gIbvwrp",
    "VrRlPtrXQg6MbAGwyB7AU+evIuwty5kLsmCBbYQXhYDzVk7yY/iMijob5GMQeN3aHt/tdM0Al4Oyd9cEL4ow+f9Wt54RHsDuBvdD",
    "DvFKmAwi0OEQSGY4NFJQwThbcvCf8dnyf1v+r8j/7R3tOwet3cP2/uF2w38J9h+r5/Tn5//2Ovvt7P73g90D2H+AJdzyf5+D/1NP",
    "YhMOGSrslQvVjckFoj/Hc6L2QFtMooWhPwJcXoTCCTR4TuWBesqheTK10M7M9aDZ/O7k7ETuXAJZQjRWbpzBNwfKBXr+L8w5VSWU",
    "1IIDMF4idAZxh8RGWqCgSWBP2lXVVzBx1TER9NWib7V168njLEXismPYqqnrTXy+vEJshgnSEzYRXvHRftxqZQPCNsffRfE0+sVv",
    "yiVmuKiCRoDAwJi1qzYHA3rAvk7zEFE/ahwawoS7BVSY+jYXCKqacc6s4gIlVzKHpSbCGvY5P1UCupjgvuwaYNMo4tuTJ99+e3aG",
    "kg/U96um6mJDLi5CnVbnQBzLZ3BL8mF3gau4npwdv35mMspshToKHQb4IgjmcFhnP4LTOPoZMb3c6QV7ZTzn+4jE3N1R54gVhYVC",
    "0xbLfWMTfQMwq+4DQl02pWutvad7zw+rK7F/uAZVe/D88Olea69eLNDZff6k1VkpcBnjjjYq0Oo8f9LZrXOH0bmEuA2s6WWFWQa5",
    "qWp303SlE0QiM3kkfKcXR0HQDPSlf+HjLkhMOdsZd9lv3NqAdzMT8FIUKImSQUCoaQ0eBL5JL3aTlApnTggIuTznRVl0IFpp5IPM",
    "WQ63IblBov6ya8wmiKmTaDYV4Vz0NPrZN3BYqdmyIIT3i+cw0ubwyg011oudMFJSplJhrR+mp6j164JdLLXlxanhLfkqODE/sL4Q",
    "1BzmLlVNqgujsBnqyyj14ZxWz2DzbZZDBjBMNe/MSzMUFaO+RkNMmBVW4dE4Iy4dFKOZ+3+Xg1KsBuBCqDjj+JbHEai1ehLdbdlQ",
    "7d44jn7RYd1B+WbLeYztRPtwigUba5m8YRQOE/qpaUZS27iMvSpcA0jr4cgMf5ZyCBnWe6/wozmmLJtD36Mu7gpHKNa4Wg4Ohc0s",
    "dHbKt4IqBKjP4tVXCBU/d30OsjDLY7qx/cs8QOC14sWH2GLsLMR31jvqOcsIzHVmmUkAtOaEUYHlAz1mDzzY1b0jPI7wL9Q0ueVS",
    "ZDYNudaeFckxDSPhhae0xpuEkhFyjCMicQw0+D3R9rTXr9MpIlY7YsnD1we6KoEAiWb9OHFnkyqnJYk7LieNGQuxGIbSBSe9xRs+",
    "+kX8NRP8hWyb8BoXSK6iKECOc/6BpEv2qiS07PGtcd/y6ym/cv0RXyJFKzaUFqDjlPCcEzAl9rZKE+oEqEUsa5rGmGflCgKZptK1",
    "squ30vLVfDADqrjZVZi/R2jA/ebuvlpqN1aR3L7JVkly4a1EX+br9DAPIz/mEBeIdYdJn+pRU1/jIA/4RAKGQCBEYM5xijDwpmSw",
    "pEW7hLQPiwPHLG6POOU63ECOkMlyRqf9OV8gaITCYeHCXRPdiq9JpGWU2LsZV/vpbJmR7bN9ts/22T7bZ/tsn+2zfbbP9tk+22f7",
    "bJ/ts322z/bZPttn+2yf7bN9ts/22T7bZ/tsn+2zfdaf/wv4a9lzAPAAAA==",
]
blob = base64.b64decode("".join(BLOB_PARTS))
tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz").extractall(".")
print("\u2705 Engine loaded.")

### 3️⃣ Your settings — edit the two lines, then click ▶

- **VIDEO_URL** — paste a YouTube link, e.g. a Flock Talk episode
- **ANTHROPIC_API_KEY** — get one at [console.anthropic.com](https://console.anthropic.com) → API Keys → Create Key (starts with `sk-ant-`)

In [ ]:
VIDEO_URL = "https://youtu.be/PASTE_YOUR_VIDEO_LINK"  #@param {type:"string"}
ANTHROPIC_API_KEY = "sk-ant-PASTE_YOUR_KEY"           #@param {type:"string"}
MAX_CLIPS = 3                                          #@param {type:"integer"}
print("✅ Saved. Video:", VIDEO_URL, "| clips:", MAX_CLIPS, "— go to step 4.")

### 4️⃣ Make the clips. Click ▶ and let it cook (5–15 min; the first run also downloads the speech model).

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["CLIP_ENGINE_ASR"] = "faster_whisper"

from pathlib import Path
from clip_engine.render import process

clips = process(source=VIDEO_URL, out_dir=Path("OUT"), work_root=Path("work"),
                mode="talk", max_clips=int(MAX_CLIPS))
print(f"\n✅ Done — {len(clips)} clips made. Go to step 5 to download.")
for c in clips: print("   •", c.name)

### 5️⃣ Download your clips. Click ▶ — `clips.zip` saves to your computer.

In [ ]:
!zip -qr clips.zip OUT
from google.colab import files
files.download("clips.zip")
print("✅ If no download started: click the 📁 folder icon on the left, right-click clips.zip → Download.")

---
**Tweak the look:** open the 📁 folder icon on the left → `config` → double-click `brand.yaml` — fonts, highlight colors, clip length all live there. Re-run step 4 after editing.

**Something errored?** Copy the red text and paste it to Claude — it built this and will fix it.